<img align='left' src = https://linea.org.br/wp-content/themes/LIneA/imagens/logo-header.jpg width=150 style='padding: 20px'> 

# PZ Compute - Input Data QA Notebook - DP0

Collection of objects made available by Rubin Observatory Legacy Survey of Space and Time (LSST), release Data Preview 0 (DP0).

Contact: Luigi Silva ([luigi.lcsilva@linea.org.br](mailto:luigi.lcsilva@linea.org.br))

<br>
<br>

LSST DESC DC2 Simulated Sky Survey: https://iopscience.iop.org/article/10.3847/1538-4365/abd62c/pdf

https://dp0-1.lsst.io/data-products-dp0-1/index.html

## Inputs and configs

fastparquet is also needed for decoding the input files.

In [ ]:
# General
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import tables_io
import psutil
import sys
import math

import json

# Bokeh
import bokeh
from bokeh.io import output_notebook, show, output_file, reset_output
from bokeh.models import ColumnDataSource, Range1d, HoverTool, CustomJSHover
from bokeh.models import CDSView, GroupFilter
from bokeh.plotting import figure, gridplot
from bokeh.transform import factor_cmap

# Astropy
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.units.quantity import Quantity

# Holoviews
import holoviews as hv
from holoviews import streams, opts
from holoviews.operation import histogram
from holoviews.operation.datashader import datashade, rasterize, shade, dynspread, spread
from holoviews.plotting.util import process_cmap

# Geoviews
import geoviews as gv
import geoviews.feature as gf
from geoviews.operation import project
from cartopy import crs
import cartopy.crs as ccrs

# Datashader
import datashader as dsh

# DB LIneA
#from dblinea import DBBase

In [ ]:
pd.set_option('display.max_rows', 5)

In [ ]:
hv.extension('bokeh')

In [ ]:
gv.extension('bokeh')

In [ ]:
output_notebook()

In [ ]:
%matplotlib inline

# Spatial distribution

In the following subsections, we will read the data of the 2D histogram corresponding to the **spatial distribution of objects**, considering their distribution in the sky according to their Right Ascension (R.A.) and Declination (DEC) coordinates, and we will make two plots.

The first plot uses the **equidistant cylindrical projection (Plate Carrée projection)**, in which the lines corresponding to R.A. are equally spaced vertical straight lines, and the lines corresponding to DEC are equally spaced horizontal straight lines. This projection distorts areas and shapes, especially at high declinations.

The second plot uses the **Mollweide projection**, an equal-area, pseudocylindrical map projection. The Mollweide projection preserves area; however, it distorts shapes, especially near the edges of the sky map. The central meridian and the celestial equator are straight lines, while other lines of R.A. and DEC are represented as curves.

Both graphs will also have a **colorbar** corresponding to the counts of objects per R.A. and DEC bin of the 2D histogram.

### How to get the data

The file containing the data of the 2D histogram was obtained using High Performance Computing (HPC) in the LIneA Apollo Cluster. There is a Python script that reads all the input Parquet files and computes the 2D histogram. In this Python file, you can personalize the R.A. and DEC bin edges. There is also an sbatch file used to submit the job to the cluster. If you are running the scripts manually by yourself in the cluster, you must change the paths in both the Python and sbatch files. Additionally, you can also change the SLURM job parameters, but, again, be careful to change these parameters in both the Python and sbatch files so that they are compatible.

Files:
1. Python script: ```histo_2d_ra_dec_compute.py```
2. Sbatch script: ```histo_2d_ra_dec_submit_job.sbatch```
3. Resulting Parquet file containing the data: ```histo_2d_ra_dec.parquet```

For running the .py script in the cluster, you must copy the files to your scratch and run:

<p style="background-color:black; color:white;">
    <font face="Courier New">
        sbatch histo_2d_ra_dec_submit_job.sbatch
    </font>
</p>

Again, be careful to **change all the paths, in the .py and .sbatch files, before running this command.**


For this Jupyter notebook, the output file ```histo_2d_ra_dec.parquet``` must be in the ```output``` folder, and this ```output``` folder must be in the same directory of the notebook itself.

**For the spatial distribution of objects, we do not compute one 2D histogram for each input Parquet file, just the total histogram considering all files together.**

## Reading the data

Below we can see the output Parquet file structure. The line 0 (type "histogram_ra_dec") contains the counts of the 2D histogram in the "values" column, and the line 1 (type "bins_ra_dec") contains the R.A. bin edges ("ra_bins") and DEC bin edges ("dec_bins") in the "values" column.

In [ ]:
### Reading the Parquet file and showing the dataframe.
df_spatial_dist = pd.read_parquet('output/histo_2d_ra_dec.parquet', engine='fastparquet')
print(df_spatial_dist)

We have to convert the lists to numpy arrays.

In [ ]:
### Generating new dataframes from the original one, containing the counts, the R.A. bin edges and the DEC bin edges, and converting
### these dataframes to numpy arrays.
histogram_ra_dec = np.array(df_spatial_dist['values'][0])
bins_ra = np.array(df_spatial_dist['values'][1]['ra_bins'])
bins_dec = np.array(df_spatial_dist['values'][1]['dec_bins'])

Now, we show some information about the R.A. bins, DEC bins and the 2D histogram counts.

In [ ]:
### Printing the information.
print("INFO - R.A. BINS")
print(f"Min. edge: {bins_ra.min():.2f} | Max. edge: {bins_ra.max():.2f} | Step: {bins_ra[1]-bins_ra[0]:.2f} | Shape: {bins_ra.shape} \n")
print("INFO - DEC BINS")
print(f"Min. edge: {bins_dec.min():.2f} | Max. edge: {bins_dec.max():.2f} | Step: {bins_dec[1]-bins_dec[0]:.2f} | Shape: {bins_dec.shape} \n") 
print("INFO- 2D HISTOGRAM COUNTS")
print(f"Min. count: {histogram_ra_dec.min()} | Max. count: {histogram_ra_dec.max()} | Shape: {histogram_ra_dec.shape}")

## Spatial distribution plot

Before making the plots, we must perform some tasks:

1. Change the 0 values in the 2D histogram counts array to NaN values, so that they appear white in the plot.
2. Compute the centers of the bins.
3. For the Plate Carrée projection, change the R.A. coordinates so that they belong to the range $[−180^{\circ},180^{\circ})$. This is necessary for inverting the x-axis in the plot, a widely used convention. We must also adjust the 2D histogram counts accordingly, so that they agree with the new R.A. range.
4. For the Mollweide projection, invert the R.A. coordinates by doing $360^{\circ} - x$ for all $x$ in the R.A. values. This is just computational artifice, which is necessary for inverting the x-axis in the plot. However, in the final plot, the R.A. and DEC ticks will be correctly showed in the original range, $[0^{\circ},360^{\circ})$. Again, we must also adjust the 2D histogram counts accordingly.
5. Transpose the 2D histogram counts arrays so that they become compatible with HoloViews/GeoViews.

In [ ]:
### Changing the 0 values to NaN values.
histogram_ra_dec_NaN = histogram_ra_dec.astype(float)
histogram_ra_dec_NaN[histogram_ra_dec_NaN == 0] = np.nan

### Getting the bins centers.
bins_ra_centers = (bins_ra[1:] + bins_ra[:-1])/2
bins_dec_centers = (bins_dec[1:] + bins_dec[:-1])/2

### Plate Carrée projection - Changing the R.A. coordinates to the range [-180,180), and changing the 2d histogram counts accordingly.
bins_ra_centers_180_range = np.where(bins_ra_centers >= 180, bins_ra_centers - 360, bins_ra_centers)
sorted_indices_180_range = np.argsort(bins_ra_centers_180_range)
histogram_ra_dec_180_range = histogram_ra_dec_NaN[sorted_indices_180_range, :]
bins_ra_centers_180_range = bins_ra_centers_180_range[sorted_indices_180_range]

### Mollweide projection - Inverting the R.A. values (360 - values), and changing the 2d histogram counts accordingly.
bins_ra_centers_inverted = np.where(bins_ra_centers <= 360, 360 - bins_ra_centers, bins_ra_centers)
sorted_indices_inverted = np.argsort(bins_ra_centers_inverted)
histogram_ra_dec_inverted = histogram_ra_dec_NaN[sorted_indices_inverted, :]
bins_ra_centers_inverted = bins_ra_centers_inverted[sorted_indices_inverted]

### Transposing the histogram arrays for the holoviews plots.
histogram_ra_dec_180_range_transpose = histogram_ra_dec_180_range.T
histogram_ra_dec_inverted_transpose = histogram_ra_dec_inverted.T

After these tasks, we are ready to make the spatial distribution plots.

First, the Plate Carrée projection plot. In this plot, **the R.A. values are in the $[−180^{\circ},180^{\circ})$ range**, where the negative values corresponds to values greater than $180^{\circ}$ in the original range, $[0^{\circ}, 360^{\circ})$.

In [ ]:
### Creating the image using holoviews.
hv_image_ra_dec = hv.Image((bins_ra_centers_180_range, bins_dec_centers, histogram_ra_dec_180_range_transpose), [f'R.A.', f'DEC'], f'Counts')

### Adjusting the image options.
hv_image_ra_dec = hv_image_ra_dec.opts(
    opts.Image(cmap='viridis', cnorm='linear', colorbar=True, width=1000, height=500,
               xlim=(180, -180), ylim=(-90, 90), tools=['hover'], clim=(10, np.nanmax(histogram_ra_dec_180_range_transpose)),
               title=f'Spatial Distribution of Objects - Plate Carrée projection', show_grid=True)
)

# Showing the graph.
hv_image_ra_dec

Second, the Mollweide projection plot. In this plot, **the R.A. values are in the original $[0^{\circ},360^{\circ})$ range**. Unfortunately, the bokeh 'hover' tool does not work with this projection.

In [ ]:
### Generating the R.A. and DEC ticks
longitudes = np.arange(30, 360, 30)
latitudes = np.arange(-75, 76, 15)

lon_labels = [f"{lon}°" for lon in longitudes]
lat_labels = [f"{lat}°" for lat in latitudes]

labels_data = {
    "lon": list(np.flip(longitudes)) + [-180] * len(latitudes),
    "lat": [0] * len(longitudes) + list(latitudes),
    "label": lon_labels + lat_labels,
}

df_labels = pd.DataFrame(labels_data)

labels_plot = gv.Labels(df_labels, kdims=["lon", "lat"], vdims=["label"]).opts(
    text_font_size="12pt",
    text_color="black",
    text_align='right',
    text_baseline='bottom',
    projection=crs.Mollweide()
)

### Creating the image using holoviews.
gv_image_ra_dec = gv.Image((bins_ra_centers_inverted, bins_dec_centers, histogram_ra_dec_inverted_transpose), [f'R.A.', f'DEC'], f'Counts')

### Doing the Mollweide projection.
gv_image_ra_dec_projected = gv.operation.project(gv_image_ra_dec, projection=crs.Mollweide())

### Generating the grid lines.
grid = gf.grid().opts(
    opts.Feature(projection=ccrs.Mollweide(), scale='110m', color='black')
)

### Adjusting the image options.
gv_image_ra_dec_projected = gv_image_ra_dec_projected.opts(cmap='viridis', cnorm='linear', colorbar=True, width=1000, height=500, 
                                                           clim=(10, np.nanmax(histogram_ra_dec_inverted_transpose)), 
                                                           title='Spatial Distribution of Objects - Mollweide projection', 
                                                           projection=ccrs.Mollweide(),  global_extent=True)

### Showing the plot.
combined_plot = gv_image_ra_dec_projected * grid * labels_plot
combined_plot

# Magnitude distributions

In the following subsections, we will read the data of the 1D histograms corresponding to the **magnitude distributions** and we will make plots for each band of the survey.

### How to get the data

The file containing the data of the 1D histogram was obtained using High Performance Computing (HPC) in the LIneA Apollo Cluster. There is a Python script that reads all the input Parquet files and computes the 1D histogram. In this Python file, you can personalize the magnitude bin edges for each band. There is also an sbatch file used to submit the job to the cluster. If you are running the scripts manually by yourself in the cluster, you must change the paths in both the Python and sbatch files. Additionally, you can also change the SLURM job parameters, but, again, be careful to change these parameters in both the Python and sbatch files so that they are compatible.

Files:
1. Python script: ```histo_1d_mag_compute.py```
2. Sbatch script: ```histo_1d_mag_submit_job.sbatch```
3. Resulting Parquet file containing the data: ```histo_1d_mag_all_bands.parquet```

For running the .py script in the cluster, you must copy the files to your scratch and run:

<p style="background-color:black; color:white;">
    <font face="Courier New">
        sbatch histo_1d_mag_submit_job.sbatch
    </font>
</p>

Again, be careful to **change all the paths, in the .py and .sbatch files, before running this command.**


For this Jupyter notebook, the output file ```histo_1d_mag_all_bands.parquet``` must be in the ```output``` folder, and this ```output``` folder must be in the same directory of the notebook itself.

**For the magnitude distributions, we compute one 1D histogram for each input Parquet file, for each band. Given a certain band, we use the same bin edges for each input file.**

## Reading the data

Below we can see the Parquet file structure. When the value of the 'filename' column is a path, the values of the 'counts' column are the 1D histogram counts for the file corresponding to this path. When the value of the 'filename' column is 'bins', the values of the 'counts' column are the bin edges used for each band.

In [ ]:
### Reading the Parquet file.
df_mag = pd.read_parquet('output/histo_1d_mag_all_bands.parquet', engine='fastparquet')

In [ ]:
### Getting the bands names and the number of bands.
bands = df_mag['band'].unique()
num_of_bands = len(bands)

### Separating the histograms of each band and the bin edges, and converting the lists to numpy arrays.
mag_histograms = {}
for band in bands:
    mag_histograms[band] = df_mag[df_mag['band'] == band]
    mag_histograms[band] = mag_histograms[band][mag_histograms[band]['filename'] != 'bins']
    mag_histograms[band] = mag_histograms[band].reset_index(drop=True)
    mag_histograms[band]['counts'] = mag_histograms[band]['counts'].apply(np.array)

mag_bins = df_mag[df_mag['filename'] == 'bins']
mag_bins = mag_bins.reset_index(drop=True)
mag_bins['counts'] = mag_bins['counts'].apply(np.array)

### Getting the input files names and the number of input files.
filenames_paths = df_mag['filename'].unique()
filenames_paths = filenames_paths[filenames_paths!='bins']
filenames_paths = filenames_paths.astype(str)
filenames = np.char.replace(filenames_paths, '/lustre/t1/cl/lsst/dp0_skinny/DP0/DP0_FULL/parquet/', '')
filenames_len = len(filenames)

In [ ]:
### PRINTING THE INFORMATION
### GENERAL INFORMATION
print(f"NUMBER OF BANDS: {num_of_bands}")
print(f"BAND NAMES: {bands}")
print(f"NUMBER OF INPUT FILES: {filenames_len}")
print(f"EACH BAND HAS A TOTAL OF {filenames_len} LINES, CORRESPONDING TO EACH INPUT FILE, PLUS ONE EXTRA LINE, CORRESPONDING TO THE BIN EDGES.\n")

### INDIVIDUAL HISTOGRAMS PREVIEW
printing_lines_for_each_band = 2
print(f"\nPRINTING {printing_lines_for_each_band} LINES FOR EACH BAND, CONTAINING THE 1D HISTOGRAM COUNTS.")
for band in bands:
    print(mag_histograms[band].head(printing_lines_for_each_band))
    print('\n')

### BIN EDGES PREVIEW
print(f"\nPRINTING THE LINES CONTAINING THE BIN EDGES (THEY CAN BE FOUND AT THE END OF THE PARQUET FILE)")
print(mag_bins.head())

### BIN EDGES INFORMATION
print("\n\nINFORMATION ABOUT THE BINS")
for band in bands:
    bin_list = mag_bins[mag_bins['band']==band]['counts'].reset_index(drop=True)[0]
    bin_min = bin_list.min()
    bin_max = bin_list.max()
    step = bin_list[1] - bin_list[0]
    shape = bin_list.shape
    print(f"Min. bin edge - band {band}: {bin_min}")
    print(f"Max. bin edge - band {band}: {bin_max}")
    print(f"Step - band {band}: {step}")
    print(f"Shape - band {band}: {shape}\n")

Now, we will compute the total 1D histogram for each band. This is done by summing the values in each bin for all the individual histograms. Below, we show some information about the total 1D histogram.

In [ ]:
### Computing the total 1D histogram.
total_mag_histograms = {}
for band in bands:
    arrays = {}
    num_of_rows = len(mag_histograms[band].index)
    for j in np.arange(0, num_of_rows, 1):
        name = 'array'+str(j)
        arrays[name] = mag_histograms[band]['counts'][j]

    somas = []

    for elementos in zip(*arrays.values()):
        soma = sum(elementos) 
        somas.append(soma)

    data = {'counts': somas,
            'bin_edges': mag_bins.loc[mag_bins['band'] == band, 'counts'].values[0][:-1]}

    total_mag_histograms[band] = pd.DataFrame(data)
    
### Printing the information.
print("INFORMATION ABOUT THE TOTAL 1D HISTOGRAM COUNTS")
for band in bands:
    print(f"Min. count - band {band}: {total_mag_histograms[band]['counts'].min()}")
    print(f"Max. count - band {band}: {total_mag_histograms[band]['counts'].max()}\n") 

## Magnitude distributions plots

### Total 1D histograms

Below, we have the plots of the total 1D histograms for each band, meaning we are considering all input parquet files together.

In [ ]:
### General settings
height = 400
width = 400

def setup_plot_data(band, histograms):
    """Sets the data to the plot for the specified band."""
    counts = histograms[band]['counts']
    bin_edges = np.array(histograms[band]['bin_edges'])
    bin_size = bin_edges[1] - bin_edges[0]
    bin_edges = np.append(bin_edges, bin_edges[-1] + bin_size)
    
    max_count_index = histograms[band]['counts'].idxmax()
    max_value = histograms[band].loc[max_count_index, 'bin_edges']
    xlim = (max_value - 5, max_value + 5)

    return counts, bin_edges, xlim

def create_histogram(band, counts, bin_edges, xlim):
    """Creates the histogram using holoviews."""
    title = f'Distribution of Magnitudes - Band {band}'
    label_name = f'mag {band}'
    magnitudes = hv.Dimension(band, label=label_name)
    mag_freqs = hv.Dimension(f'{band}_freqs', label=f'{label_name} freqs')
    
    return hv.Histogram((counts, bin_edges), kdims=magnitudes, vdims=mag_freqs).opts(
        title=title, xlabel=label_name, ylabel='frequencies', height=height, width=width, xlim=xlim)

### Preparing the histograms data.
mag_distribution_histo = {}
for band in bands:
    band = band.lower()
    counts, bin_edges, xlim = setup_plot_data(band, total_mag_histograms)
    mag_distribution_histo[band] = create_histogram(band, counts, bin_edges, xlim)

### Composing the plots in a hv.Layout and showing.
mag_distribution = hv.Layout([mag_distribution_histo[band] for band in bands]).cols(2)
mag_distribution

### Individual 1D histograms for each input file

Now, in the plots below, for a given band, each light-colored curve was obtained from the 1D histogram of each individual Parquet file. The red curve, in turn, represents the mean value of counts in each bin, considering all input Parquet files.

Note that if the histograms from each individual input file are very similar, meaning each input file has enough data so that statistical deviations are very small, **you may need to zoom in on the plot to distinguish the light curves, as they will all be very close to the average curve (red curve)**.

In [ ]:
# Função ajustada para criar gráficos para um dataframe específico e banda
def create_plot(df, bins, title):
    curves = []
    all_counts = []

    for i, row in df.iterrows():
        counts = np.array(row['counts'])
        centers = (bins[:-1] + bins[1:]) / 2
        curve = hv.Curve((centers, counts), 'Magnitude', 'Counts').opts(line_width=2, alpha=0.3)
        curves.append(curve)
        all_counts.append(counts)

    mean_counts = np.mean(all_counts, axis=0)
    mean_curve = hv.Curve((centers, mean_counts), 'Magnitude', 'Counts').opts(line_width=4, color='red').relabel('Mean Counts')

    overlay = hv.Overlay(curves + [mean_curve]).opts(
        hv.opts.Overlay(title=title, width=400, height=400, legend_position='top_left', show_legend=True, legend_limit=100),
    )
    return overlay

# Criando gráficos para todas as bandas
plots = []
for band in bands:
    band_df = mag_histograms[band]
    bins = np.array(mag_bins.loc[mag_bins['band'] == band, 'counts'].values[0])
    plot = create_plot(band_df, bins, f"Individual Files - Mag. Distrib. - Band {band}")
    plots.append(plot)

# Organizando todos os gráficos em um layout
layout = hv.Layout(plots).opts(opts.Layout(shared_axes=False)).cols(2)

layout

# Magnitude x Magnitude Error

## Reading the data

First, let us read the data.

In [ ]:
df_mag_magerr = pd.read_parquet('output/histo_2d_mag_magerr_all_bands.parquet')

In [ ]:
df_mag_magerr.head(5)

Now, let us separate the data of 'histogram' type and of 'bins' type.

In [ ]:
mag_magerr_histograms = {}
for band in bands:
    mag_magerr_histograms[band] = df_mag_magerr[df_mag_magerr['band'] == band]
    mag_magerr_histograms[band] = mag_magerr_histograms[band][mag_magerr_histograms[band]['type'] != 'bins']
    mag_magerr_histograms[band] = mag_magerr_histograms[band].reset_index(drop=True)

mag_bins = df_mag_magerr[df_mag_magerr['type'] == 'bins']
mag_bins = mag_bins.reset_index(drop=True)

In [ ]:
mag_magerr_histograms['u']

In [ ]:
mag_bins.head(5)

Now, let us decode the data, so that the histograms binary strings became lists and the bins binary strings became dictionaries.

In [ ]:
### HISTOGRAMS
for band in bands:
    num_of_rows = len(mag_magerr_histograms[band].index)
    for j in range(num_of_rows):
        # Decodificar a string binária e carregar como JSON
        decoded_json = mag_magerr_histograms[band].at[j, 'values'].decode('utf-8')
        # Converter a lista resultante em um array numpy
        array = np.array(json.loads(decoded_json))
        # Atualizar o DataFrame com o array numpy
        mag_magerr_histograms[band].at[j, 'values'] = array
        
### BINS
decoded_dicts = [json.loads(value.decode("utf-8")) for value in mag_bins['values']]
mag_bins['values'] = decoded_dicts

In [ ]:
mag_magerr_histograms['i']

In [ ]:
mag_bins.head(5)

## Magnitude x Magnitude Error plots

Getting the center of the bin edges for plotting:

In [ ]:
x_edges = {}
y_edges = {}

for band in bands:
    mag_bins_vals = np.array(mag_bins.loc[mag_bins['band'] == band, 'values'].values[0]['mag_bins'])
    magerr_bins_vals = np.array(mag_bins.loc[mag_bins['band'] == band, 'values'].values[0]['magerr_bins'])
    
    x_edges[band] = (mag_bins_vals[1:] + mag_bins_vals[:-1])/2
    y_edges[band] = (magerr_bins_vals[1:] +  magerr_bins_vals[:-1])/2

Converting the histogram to a 2D np.array and transposing it for the plots:

In [ ]:
histo = {}

for band in bands:
    histo[band] = np.vstack(mag_magerr_histograms[band]['values'])   
    histo[band] = histo[band].T

Making the plot:

In [ ]:
%%time
plots = []

for band in bands:
    hv_image = hv.Image((x_edges[band], y_edges[band], histo[band]), [f'mag_{band}', f'magerr_{band}'], f'counts_{band}')
    
    hv_image = hv_image.opts(
        opts.Image(cmap='viridis', cnorm='log', colorbar=True, width=600, height=400,
                   xlim=(20, 27), ylim=(0, 0.2), tools=['hover'], clim=(10, np.max(histo[band])),
                   title=f'Magnitude x Magnitude Error - Band {band}')
    )
    
    plots.append(hv_image)

# Organizando os gráficos em duas colunas
layout = hv.Layout(plots).cols(2)
layout